#  Basic Tasks

In [0]:
%sql
create catalog if not exists cyntexa_dev;

In [0]:
%sql
create schema if not exists cyntexa_dev.sales; 

In [0]:
%sql
CREATE OR REPLACE TABLE cyntexa_dev.sales.orders_raw
(
    order_id INT,
    customer_id STRING,
    customer_name STRING,
    email STRING,
    order_amount DECIMAL(10,2),
    order_status STRING,
    order_date DATE
);

In [0]:
%sql
INSERT INTO cyntexa_dev.sales.orders_raw VALUES
(1,'CUST1001','Rahul','rahul@gmail.com',2500,'Completed','2026-08-01'),
(2,'CUST1002','Aman','aman@gmail.com',1800,'Pending','2026-08-02'),
(3,'CUST1003','Neha','neha@gmail.com',3200,'Completed','2026-08-03'),
(4,'CUST1004','Priya','priya@gmail.com',1500,'Cancelled','2026-08-04'),
(5,'CUST1005','Karan','karan@gmail.com',4200,'Completed','2026-08-05'),
(6,'CUST1001','Rahul','rahul@gmail.com',900,'Completed','2026-08-06'),
(7,'CUST1006','Riya','riya@gmail.com',2750,'Pending','2026-08-07'),
(8,'CUST1007','Mohit','mohit@gmail.com',3100,'Completed','2026-08-08'),
(9,'CUST1003','Neha','neha@gmail.com',1100,'Completed','2026-08-09'),
(10,'CUST1008','Vikas','vikas@gmail.com',5000,'Completed','2026-08-10');

In [0]:
%sql
CREATE OR REPLACE VIEW cyntexa_dev.sales.orders_view AS
SELECT *
FROM cyntexa_dev.sales.orders_raw
WHERE order_status = 'Completed';

In [0]:
%sql
select * from samples.tpch.customer
limit 10;

In [0]:
%sql
SELECT
    c_nationkey,
    COUNT(*) AS total_customers
FROM samples.tpch.customer
GROUP BY c_nationkey
ORDER BY total_customers DESC;

In [0]:
%sql
SELECT
    c_acctbal,
    c_name
FROM samples.tpch.customer
ORDER BY c_acctbal DESC
LIMIT 5;

# **INTERMEDIATE TASK**

In [0]:
%sql
SELECT customer_id,
       CONCAT(left(customer_id,4), '****') maskked_customer_id
       from cyntexa_dev.sales.orders_raw

In [0]:
%sql
create or replace function mask_customer(id string)
returns string
return concat(left(id, 4), "****")

In [0]:
%sql
select customer_name, customer_id, mask_customer(customer_id) as masked_id 
from cyntexa_dev.sales.orders_raw
 

In [0]:
%sql
create or replace function mask_email(email string)
returns string
return concat("****@",  SPLIT(email,'@')[1]);

In [0]:
%sql
select customer_name, email, mask_email(email) as masked_email
from cyntexa_dev.sales.orders_raw

In [0]:
%sql
CREATE OR REPLACE TABLE cyntexa_dev.sales.customers
(
customer_id STRING,
city STRING
);

In [0]:
%sql
INSERT INTO cyntexa_dev.sales.customers VALUES
('CUST1001','Jaipur'),
('CUST1002','Delhi'),
('CUST1003','Mumbai'),
('CUST1004','Pune'),
('CUST1005','Noida'),
('CUST1006','Indore'),
('CUST1007','Udaipur'),
('CUST1008','Surat');

In [0]:
%sql
CREATE OR REPLACE VIEW cyntexa_dev.sales.customer_spend AS

SELECT
c.customer_id,
o.customer_name,
c.city,
SUM(o.order_amount) total_spend

FROM cyntexa_dev.sales.orders_view o
JOIN cyntexa_dev.sales.customers c
ON o.customer_id=c.customer_id

GROUP BY
c.customer_id,
o.customer_name,
c.city;

In [0]:
%sql
drop view customer_spend

In [0]:
%sql
SELECT * FROM cyntexa_dev.sales.customer_spend
ORDER BY total_spend DESC;

# Advanced Tasks 

# Q. 8

### We separate catalogs by environment (Dev, Staging, Production) to isolate development and production data. Schemas represent business domains such as Sales, Finance, and HR. This structure improves governance, security, and deployment using Unity Catalog.

# Q. 9

## Columns to Mask

### Column  ----->      Why

email      ------->      PII

customer_id ------>       Sensitive identifier

phone      ------>        Personal data

## Role Access

### Role --> Access

Data Engineer  --> Unmasked

Data Analyst  -->  Masked

Business User  --> Masked only

### Customer email, phone, and customer ID should be masked because they contain personally identifiable information (PII). Data Engineers receive full access, while Analysts query masked views created with SQL UDFs. Unity Catalog grants SELECT permission only on the masked view for analyst roles.

# Q. 10

In [0]:
%sql
WITH revenue_cte AS
(
SELECT
    c.c_nationkey AS region,
    c.c_custkey,
    c.c_name,
    SUM(o.o_totalprice) revenue

FROM samples.tpch.customer c

JOIN samples.tpch.orders o
ON c.c_custkey=o.o_custkey

GROUP BY
    c.c_nationkey,
    c.c_custkey,
    c.c_name
),

ranked AS
(
SELECT *,
DENSE_RANK() OVER
(
PARTITION BY region
ORDER BY revenue DESC
) rank
FROM revenue_cte
)

SELECT *
FROM ranked
WHERE rank<=5
ORDER BY region,rank;